# 💬 ASSIGNMENT NLP – 2: Sentiment Analysis using NLP Pipeline & ML Models

**Internship:** Data Science Internship – February 2026  
**Task:** Build an end-to-end Sentiment Analysis system using NLP preprocessing, feature engineering, and multiple ML models.

---

## 📌 Objective
Process raw IMDB movie review text through a complete NLP pipeline, convert it into ML-ready features using BoW and TF-IDF, train multiple classifiers, and compare their performance.

---

## 🗂️ Table of Contents
1. [Install & Import Libraries](#1)
2. [Task 1 — Data Understanding & EDA](#2)
3. [Task 2 — NLP Preprocessing Pipeline](#3)
4. [Task 3 — Feature Engineering (BoW & TF-IDF)](#4)
5. [Task 4 — Model Building](#5)
6. [Task 5 — Model Evaluation](#6)
7. [Task 6 — Comparison & Insights](#7)
8. [Summary of Findings](#8)

---
## 1. Install & Import Libraries <a id='1'></a>

In [ ]:
# Install all required packages
# nltk  : NLP preprocessing (stopwords, tokenization, stemming, lemmatization)
# datasets : Hugging Face library to load IMDB without manual download
!pip install nltk datasets scikit-learn matplotlib seaborn wordcloud --quiet

In [ ]:
# ─────────────────────────────────────────────────────────────
# STANDARD LIBRARY & DATA HANDLING
# ─────────────────────────────────────────────────────────────
import re                          # Regular expressions for text cleaning
import time
import warnings
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd

# ─────────────────────────────────────────────────────────────
# VISUALISATION
# ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud     # Generate word-cloud images

# ─────────────────────────────────────────────────────────────
# NLTK — NLP PREPROCESSING TOOLS
# ─────────────────────────────────────────────────────────────
import nltk

# Download required NLTK resources (runs once, cached afterwards)
nltk.download("punkt",        quiet=True)   # Sentence/word tokenizer
nltk.download("punkt_tab",   quiet=True)   # Updated punkt data
nltk.download("stopwords",   quiet=True)   # Common English stopwords
nltk.download("wordnet",     quiet=True)   # WordNet lexical database (for lemmatizer)
nltk.download("omw-1.4",     quiet=True)   # Open Multilingual WordNet (lemmatizer dependency)

from nltk.corpus    import stopwords        # English stop-word list
from nltk.tokenize  import word_tokenize    # Tokenizer
from nltk.stem      import PorterStemmer    # Stemmer  (fast, rule-based)
from nltk.stem      import WordNetLemmatizer# Lemmatizer (dictionary-based, slower but cleaner)

# ─────────────────────────────────────────────────────────────
# SCIKIT-LEARN — FEATURE ENGINEERING & ML MODELS
# ─────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection         import train_test_split, cross_val_score
from sklearn.linear_model            import LogisticRegression
from sklearn.naive_bayes             import MultinomialNB
from sklearn.tree                    import DecisionTreeClassifier
from sklearn.ensemble                import RandomForestClassifier   # Optional bonus
from sklearn.metrics                 import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline     # Chain vectorizer + model cleanly

# Dataset loader
from datasets import load_dataset

# ─────────────────────────────────────────────────────────────
# REPRODUCIBILITY SEED
# ─────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

print("✅ All libraries imported successfully!")

---
## Task 1 — Data Understanding & EDA <a id='2'></a>

### Dataset: IMDB Movie Reviews
- **Source:** Hugging Face `datasets` (identical to the Kaggle IMDB dataset)
- **Size:** 50,000 reviews (25,000 train + 25,000 test)
- **Labels:** Binary — `0 = Negative`, `1 = Positive`
- **Task:** Predict sentiment from raw review text

In [ ]:
# ─────────────────────────────────────────────────────────────
# LOAD DATASET
# ─────────────────────────────────────────────────────────────

print("⏳ Loading IMDB dataset...")
raw = load_dataset("imdb")   # Downloads and caches automatically

# Convert to pandas DataFrames for easy manipulation
train_df = pd.DataFrame(raw["train"])
test_df  = pd.DataFrame(raw["test"])

# Rename label column for clarity
train_df["sentiment"] = train_df["label"].map({0: "negative", 1: "positive"})
test_df["sentiment"]  = test_df["label"].map({0: "negative", 1: "positive"})

print("✅ Dataset loaded!")
print(f"   → Training samples : {len(train_df):,}")
print(f"   → Test samples     : {len(test_df):,}")
print(f"   → Columns          : {list(train_df.columns)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATORY DATA ANALYSIS (EDA)
# ─────────────────────────────────────────────────────────────

# --- 1. Preview first few rows ---
print("── Sample rows ──")
display(train_df[["text", "sentiment"]].head(4))

# --- 2. Class distribution ---
print("\n── Sentiment distribution (Train) ──")
print(train_df["sentiment"].value_counts())

# --- 3. Review length statistics ---
train_df["word_count"] = train_df["text"].apply(lambda x: len(x.split()))
train_df["char_count"] = train_df["text"].apply(len)

print("\n── Word count statistics ──")
print(train_df.groupby("sentiment")["word_count"].describe().round(1))

# --- 4. Plots ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 4a. Class distribution bar chart
counts = train_df["sentiment"].value_counts()
axes[0].bar(counts.index, counts.values,
            color=["#e74c3c", "#2ecc71"], edgecolor="black")
axes[0].set_title("Sentiment Distribution (Train)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 150, f"{v:,}", ha="center", fontweight="bold")

# 4b. Word count distribution per class
for label, color in [("positive", "#2ecc71"), ("negative", "#e74c3c")]:
    axes[1].hist(
        train_df[train_df["sentiment"] == label]["word_count"],
        bins=60, alpha=0.6, color=color, label=label, edgecolor="none"
    )
axes[1].set_title("Word Count Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Number of Words")
axes[1].set_ylabel("Frequency")
axes[1].legend()

# 4c. Average word count boxplot
axes[2].boxplot(
    [train_df[train_df["sentiment"]=="positive"]["word_count"],
     train_df[train_df["sentiment"]=="negative"]["word_count"]],
    labels=["Positive", "Negative"],
    patch_artist=True,
    boxprops=dict(facecolor="#85c1e9", color="black")
)
axes[2].set_title("Word Count Boxplot", fontsize=13, fontweight="bold")
axes[2].set_ylabel("Number of Words")

plt.suptitle("IMDB Dataset — Exploratory Data Analysis",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("eda_plots.png", dpi=120, bbox_inches="tight")
plt.show()
print("📊 EDA plots saved.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# WORD CLOUDS — Positive vs Negative reviews
# ─────────────────────────────────────────────────────────────

# Combine all text for each class
pos_text = " ".join(train_df[train_df["sentiment"]=="positive"]["text"].tolist())
neg_text = " ".join(train_df[train_df["sentiment"]=="negative"]["text"].tolist())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, text, title, color in [
    (axes[0], pos_text, "☀️  Positive Reviews", "Greens"),
    (axes[1], neg_text, "🌧  Negative Reviews",  "Reds")
]:
    # WordCloud object: generates frequency-weighted word images
    wc = WordCloud(
        width=800, height=400,
        background_color="white",
        colormap=color,
        max_words=100,
        stopwords=set(stopwords.words("english")),
        collocations=False    # Avoid repeated bigrams
    ).generate(text)

    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title, fontsize=14, fontweight="bold")

plt.suptitle("Word Clouds — Top Words by Sentiment",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("wordclouds.png", dpi=120, bbox_inches="tight")
plt.show()
print("☁️  Word clouds saved.")

---
## Task 2 — NLP Preprocessing Pipeline <a id='3'></a>

A reusable preprocessing function applies the following steps in sequence:

| Step | Why it matters |
|---|---|
| Remove HTML tags | IMDB reviews contain `<br />` and other HTML artifacts |
| Remove URLs | URLs add noise with no sentiment signal |
| Remove special characters | Punctuation and digits rarely carry sentiment meaning |
| Lowercase | Ensures "Good" and "good" map to the same token |
| Tokenization | Splits sentences into individual word units |
| Stopword removal | Eliminates high-frequency, low-information words ("the", "is") |
| Lemmatization | Reduces words to their dictionary base form ("running" → "run") |
| Stemming (optional) | More aggressive truncation ("running" → "run", "studies" → "studi") |

In [ ]:
# ─────────────────────────────────────────────────────────────
# NLP PREPROCESSING — REUSABLE FUNCTIONS
# ─────────────────────────────────────────────────────────────

# Initialise NLP tools once (expensive to create repeatedly)
STOP_WORDS  = set(stopwords.words("english"))   # 179 common English stopwords
LEMMATIZER  = WordNetLemmatizer()                # WordNet-based lemmatizer
STEMMER     = PorterStemmer()                    # Porter stemmer (faster, less accurate)


def remove_html(text: str) -> str:
    """Strip HTML tags e.g. <br />, <b>, <i> from text."""
    return re.sub(r"<[^>]+>", " ", text)


def remove_urls(text: str) -> str:
    """Remove web URLs (http/https/www)."""
    return re.sub(r"http\S+|www\.\S+", " ", text)


def remove_special_chars(text: str) -> str:
    """Remove non-alphabetic characters; keep spaces."""
    return re.sub(r"[^a-zA-Z\s]", " ", text)


def normalize_whitespace(text: str) -> str:
    """Collapse multiple spaces into one and strip edges."""
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text: str) -> list:
    """Split text into individual word tokens using NLTK word_tokenize."""
    return word_tokenize(text)


def remove_stopwords(tokens: list) -> list:
    """
    Filter out stopwords and very short tokens (length <= 2).
    Short tokens like 'br', 'nt', 'im' carry little information.
    """
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 2]


def lemmatize_tokens(tokens: list) -> list:
    """
    Convert each token to its base (lemma) form.
    Example: 'running' → 'run', 'better' → 'good'
    Lemmatization is preferred over stemming because it produces
    real dictionary words.
    """
    return [LEMMATIZER.lemmatize(t) for t in tokens]


def stem_tokens(tokens: list) -> list:
    """
    Apply Porter stemming to each token.
    Example: 'running' → 'run', 'studies' → 'studi'
    Faster but produces non-dictionary fragments.
    """
    return [STEMMER.stem(t) for t in tokens]


def preprocess_text(text: str, use_stemming: bool = False) -> str:
    """
    Complete NLP preprocessing pipeline.

    Steps applied (in order):
      1. Remove HTML tags
      2. Remove URLs
      3. Remove special characters / digits
      4. Lowercase
      5. Normalise whitespace
      6. Tokenize into words
      7. Remove stopwords and short tokens
      8. Lemmatize (or stem if use_stemming=True)
      9. Re-join tokens into a single cleaned string

    Parameters
    ----------
    text         : str  – Raw review text.
    use_stemming : bool – If True, apply stemming instead of lemmatization.

    Returns
    -------
    str – Fully preprocessed text.
    """
    # Steps 1–4: structural cleaning + lowercase
    text = remove_html(text)
    text = remove_urls(text)
    text = remove_special_chars(text)
    text = text.lower()
    text = normalize_whitespace(text)

    # Step 5: Tokenization
    tokens = tokenize(text)

    # Step 6: Stopword removal
    tokens = remove_stopwords(tokens)

    # Step 7: Lemmatize or Stem
    if use_stemming:
        tokens = stem_tokens(tokens)
    else:
        tokens = lemmatize_tokens(tokens)

    # Step 8: Re-join tokens into a string for vectorizers
    return " ".join(tokens)


print("✅ All preprocessing functions defined.")

# ── Demonstrate the pipeline on one review ────────────────────
sample_raw = train_df["text"].iloc[0]
sample_clean = preprocess_text(sample_raw)

print("\n── Preprocessing Demo ──")
print(f"ORIGINAL ({len(sample_raw)} chars):")
print(sample_raw[:300], "...")
print(f"\nCLEANED ({len(sample_clean)} chars):")
print(sample_clean[:300], "...")

In [ ]:
# ─────────────────────────────────────────────────────────────
# APPLY PREPROCESSING TO FULL DATASET
# We work on a 10,000-sample subset to keep runtime reasonable.
# Remove the slicing below to run on all 50,000 samples.
# ─────────────────────────────────────────────────────────────

SAMPLE_SIZE = 10000   # Change to len(train_df) for full dataset

# Stratified sample: 5,000 positive + 5,000 negative
df = train_df.groupby("sentiment", group_keys=False).apply(
    lambda x: x.sample(SAMPLE_SIZE // 2, random_state=SEED)
).reset_index(drop=True)

print(f"⏳ Preprocessing {SAMPLE_SIZE:,} reviews... (this may take ~1 minute)")
t0 = time.time()

# Apply the pipeline to every review (using lemmatization by default)
df["clean_text"] = df["text"].apply(preprocess_text)

elapsed = time.time() - t0
print(f"✅ Preprocessing complete in {elapsed:.1f}s")
print(f"   → Original avg length  : {df['text'].apply(len).mean():.0f} chars")
print(f"   → Cleaned avg length   : {df['clean_text'].apply(len).mean():.0f} chars")

# Check for empty strings after cleaning (edge case)
empty = (df["clean_text"].str.strip() == "").sum()
print(f"   → Empty reviews after cleaning: {empty}")
if empty > 0:
    df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)
    print(f"   → Removed {empty} empty rows.")

display(df[["text", "clean_text", "sentiment"]].head(3))

---
## Task 3 — Feature Engineering (BoW & TF-IDF) <a id='4'></a>

### 📐 Two Vectorization Strategies

| Method | How it works | Strength | Weakness |
|---|---|---|---|
| **Bag of Words (BoW)** | Counts raw word frequency in each document | Simple, fast, interpretable | Ignores word importance across corpus |
| **TF-IDF** | Weights words by frequency × inverse document frequency | Downweights common words; emphasises distinctive words | Slightly more complex |

In [ ]:
# ─────────────────────────────────────────────────────────────
# TRAIN / VALIDATION / TEST SPLIT
# ─────────────────────────────────────────────────────────────

# Map string labels → binary integers for sklearn
df["label_id"] = df["sentiment"].map({"positive": 1, "negative": 0})

X = df["clean_text"].values   # Cleaned text array
y = df["label_id"].values     # Binary label array

# 70% train | 15% validation | 15% test  (two-stage split)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("✅ Data split complete:")
print(f"   → Train : {len(X_train):,} samples")
print(f"   → Val   : {len(X_val):,} samples")
print(f"   → Test  : {len(X_test):,} samples")

In [ ]:
# ─────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# Both vectorizers are fitted ONLY on training data to prevent
# data leakage (test vocabulary must not influence training).
# ─────────────────────────────────────────────────────────────

# ── Bag of Words (CountVectorizer) ────────────────────────────
# max_features: keep only the top 10,000 most frequent words
# ngram_range : use unigrams and bigrams ("not good" as one feature)
bow_vectorizer = CountVectorizer(
    max_features = 10000,
    ngram_range  = (1, 2),   # Unigrams + bigrams
    min_df       = 2         # Ignore terms that appear in < 2 documents
)

# Fit on training data only, then transform all splits
X_train_bow = bow_vectorizer.fit_transform(X_train)   # Learn vocabulary + encode
X_val_bow   = bow_vectorizer.transform(X_val)          # Encode using learned vocab
X_test_bow  = bow_vectorizer.transform(X_test)         # Encode using learned vocab

print("── Bag of Words ──")
print(f"   Vocabulary size  : {len(bow_vectorizer.vocabulary_):,}")
print(f"   Train matrix     : {X_train_bow.shape}  (sparse)")

# ── TF-IDF ───────────────────────────────────────────────────
# sublinear_tf: apply log(1 + tf) to dampen the effect of very
#               frequent terms (e.g., a word appearing 100 times
#               is not 100× more important than appearing 1 time)
tfidf_vectorizer = TfidfVectorizer(
    max_features = 10000,
    ngram_range  = (1, 2),
    min_df       = 2,
    sublinear_tf = True      # log-transform term frequency
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf   = tfidf_vectorizer.transform(X_val)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

print("\n── TF-IDF ──")
print(f"   Vocabulary size  : {len(tfidf_vectorizer.vocabulary_):,}")
print(f"   Train matrix     : {X_train_tfidf.shape}  (sparse)")

# ── Top features by TF-IDF weight ────────────────────────────
tfidf_df = pd.DataFrame({
    "Feature" : tfidf_vectorizer.get_feature_names_out(),
    "Score"   : np.asarray(X_train_tfidf.mean(axis=0)).flatten()
}).sort_values("Score", ascending=False)

print("\n── Top 15 TF-IDF features (unigrams) ──")
top_uni = tfidf_df[~tfidf_df["Feature"].str.contains(" ")].head(15)
print(top_uni.to_string(index=False))

In [ ]:
# ── Visualise top BoW and TF-IDF features ────────────────────

bow_df = pd.DataFrame({
    "Feature" : bow_vectorizer.get_feature_names_out(),
    "Score"   : np.asarray(X_train_bow.mean(axis=0)).flatten()
}).sort_values("Score", ascending=False)

top_bow   = bow_df[~bow_df["Feature"].str.contains(" ")].head(15)
top_tfidf = tfidf_df[~tfidf_df["Feature"].str.contains(" ")].head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, data, title, color in [
    (axes[0], top_bow,   "Top 15 Words — Bag of Words", "#3498db"),
    (axes[1], top_tfidf, "Top 15 Words — TF-IDF",       "#e67e22")
]:
    ax.barh(data["Feature"][::-1], data["Score"][::-1],
            color=color, edgecolor="black")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Average Score")

plt.suptitle("Feature Importance: BoW vs TF-IDF",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Task 4 — Model Building <a id='5'></a>

Four models are trained on both BoW and TF-IDF features:

| Model | Key Characteristic |
|---|---|
| **Logistic Regression** | Linear classifier; strong baseline for text; interpretable coefficients |
| **Multinomial Naive Bayes** | Probabilistic; assumes feature independence; fast; works well with BoW |
| **Decision Tree** | Non-linear; creates binary decision rules; prone to overfitting on text |
| **Random Forest** *(Bonus)* | Ensemble of trees; reduces overfitting via averaging |

> **Note:** All models are trained on both BoW and TF-IDF, giving **8 model-vectorizer combinations** in total.

In [ ]:
# ─────────────────────────────────────────────────────────────
# TASK 4: MODEL BUILDING & TASK 5: EVALUATION (combined)
# ─────────────────────────────────────────────────────────────

# ── Model definitions ─────────────────────────────────────────
MODELS = {
    "Logistic Regression" : LogisticRegression(
        max_iter   = 500,       # Enough iterations to converge
        C          = 1.0,       # Regularisation strength (default)
        solver     = "lbfgs",   # Efficient for medium datasets
        random_state = SEED
    ),
    "Naive Bayes" : MultinomialNB(
        alpha = 1.0             # Laplace smoothing (prevents zero probabilities)
    ),
    "Decision Tree" : DecisionTreeClassifier(
        max_depth    = 20,      # Limit depth to reduce overfitting
        min_samples_leaf = 5,   # Require at least 5 samples at leaf nodes
        random_state = SEED
    ),
    "Random Forest" : RandomForestClassifier(
        n_estimators = 100,     # 100 decision trees in the ensemble
        max_depth    = 20,
        random_state = SEED,
        n_jobs       = -1       # Use all CPU cores
    )
}

# ── Vectorized feature sets ───────────────────────────────────
FEATURE_SETS = {
    "BoW"   : (X_train_bow,   X_val_bow,   X_test_bow),
    "TF-IDF": (X_train_tfidf, X_val_tfidf, X_test_tfidf)
}


def train_and_evaluate(model_name: str, model,
                        feat_name: str,
                        X_tr, X_vl, X_ts,
                        y_tr, y_vl, y_ts) -> dict:
    """
    Train a model and compute evaluation metrics on both
    validation and test sets.

    Parameters
    ----------
    model_name : str   – Display name of the ML algorithm.
    model      :       – Sklearn estimator (unfitted).
    feat_name  : str   – Feature set name ('BoW' or 'TF-IDF').
    X_tr/vl/ts : sparse matrix – Feature matrices.
    y_tr/vl/ts : array         – Label arrays.

    Returns
    -------
    dict with model info and all computed metrics.
    """
    t0 = time.time()

    # ── Train on training split ───────────────────────────────
    model.fit(X_tr, y_tr)

    # ── Predict on test split ─────────────────────────────────
    y_pred = model.predict(X_ts)

    # ── Compute metrics ───────────────────────────────────────
    acc  = accuracy_score(y_ts, y_pred)
    prec = precision_score(y_ts, y_pred, average="binary", zero_division=0)
    rec  = recall_score(y_ts, y_pred, average="binary",    zero_division=0)
    f1   = f1_score(y_ts, y_pred, average="binary",        zero_division=0)

    elapsed = time.time() - t0

    result = {
        "Model"     : model_name,
        "Features"  : feat_name,
        "Accuracy"  : round(acc,  4),
        "Precision" : round(prec, 4),
        "Recall"    : round(rec,  4),
        "F1 Score"  : round(f1,   4),
        "Time (s)"  : round(elapsed, 2),
        "_model"    : model,
        "_y_pred"   : y_pred,
        "_y_true"   : y_ts
    }

    print(f"  ✅ {model_name:22s} [{feat_name:6s}] | "
          f"Acc: {acc:.4f} | Prec: {prec:.4f} | "
          f"Rec: {rec:.4f} | F1: {f1:.4f} | {elapsed:.1f}s")

    return result


# ── Run all model × feature combinations ─────────────────────
print("="*78)
print("  🚀  TRAINING ALL MODELS (4 models × 2 feature sets = 8 combinations)")
print("="*78)

all_results = []

for feat_name, (X_tr, X_vl, X_ts) in FEATURE_SETS.items():
    print(f"\n── Feature Set: {feat_name} ──")
    for model_name, model in MODELS.items():
        result = train_and_evaluate(
            model_name, model, feat_name,
            X_tr, X_vl, X_ts,
            y_train, y_val, y_test
        )
        all_results.append(result)

print("\n✅ All models trained and evaluated!")

---
## Task 5 — Model Evaluation <a id='6'></a>

In [ ]:
# ─────────────────────────────────────────────────────────────
# RESULTS TABLE
# ─────────────────────────────────────────────────────────────

# Build a clean summary DataFrame (drop internal columns)
results_df = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith("_")}
    for r in all_results
]).set_index(["Model", "Features"])

print("\n" + "="*70)
print("          📊  MODEL EVALUATION RESULTS (Test Set)")
print("="*70)

display(
    results_df.style
    .format("{:.4f}", subset=["Accuracy", "Precision", "Recall", "F1 Score"])
    .highlight_max(axis=0, color="#d4edda",
                   subset=["Accuracy", "Precision", "Recall", "F1 Score"])
    .highlight_min(axis=0, color="#f8d7da",
                   subset=["Accuracy", "Precision", "Recall", "F1 Score"])
    .set_caption("🟢 Green = Best  |  🔴 Red = Worst  (per column)")
)

# Best model overall
best_idx = results_df["F1 Score"].idxmax()
best_f1  = results_df["F1 Score"].max()
print(f"\n🏆 Best model: {best_idx} — F1 Score: {best_f1:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# GROUPED BAR CHART — All metrics across all model×feature combos
# ─────────────────────────────────────────────────────────────

metrics = ["Accuracy", "Precision", "Recall", "F1 Score"]
model_names = list(MODELS.keys())
feature_names = list(FEATURE_SETS.keys())

fig, axes = plt.subplots(1, len(metrics), figsize=(20, 5), sharey=True)
colors = {"BoW": "#3498db", "TF-IDF": "#e67e22"}
x      = np.arange(len(model_names))
width  = 0.35

for ax, metric in zip(axes, metrics):
    for i, feat in enumerate(feature_names):
        vals = [
            results_df.loc[(m, feat), metric]
            for m in model_names
        ]
        offset = (i - 0.5) * width
        bars   = ax.bar(x + offset, vals, width,
                        label=feat, color=colors[feat], edgecolor="black")
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.003,
                    f"{val:.3f}",
                    ha="center", va="bottom", fontsize=7)

    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(" ", "\n") for m in model_names], fontsize=9)
    ax.set_ylim(0.50, 1.05)
    ax.legend(fontsize=9)

fig.suptitle("Model Performance: BoW vs TF-IDF (Test Set)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFUSION MATRICES — Best model per feature set
# ─────────────────────────────────────────────────────────────

class_names = ["Negative", "Positive"]

# Find best result per feature type (by F1)
best_bow   = max([r for r in all_results if r["Features"]=="BoW"],
                 key=lambda r: r["F1 Score"])
best_tfidf = max([r for r in all_results if r["Features"]=="TF-IDF"],
                 key=lambda r: r["F1 Score"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, result in [(axes[0], best_bow), (axes[1], best_tfidf)]:
    cm      = confusion_matrix(result["_y_true"], result["_y_pred"])
    cm_pct  = cm.astype(float) / cm.sum() * 100
    annot   = np.array([
        [f"{v}\n({p:.1f}%)" for v, p in zip(row_v, row_p)]
        for row_v, row_p in zip(cm, cm_pct)
    ])
    sns.heatmap(
        cm, annot=annot, fmt="", cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=0.5, linecolor="grey", ax=ax
    )
    ax.set_title(
        f"Best {result['Features']} Model:\n"
        f"{result['Model']} (F1={result['F1 Score']:.4f})",
        fontsize=11, fontweight="bold"
    )
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

    # Full classification report
    print(f"\n── {result['Model']} [{result['Features']}] — Classification Report ──")
    print(classification_report(
        result["_y_true"], result["_y_pred"],
        target_names=class_names
    ))

fig.suptitle("Confusion Matrices — Best BoW & TF-IDF Models",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# TOP POSITIVE & NEGATIVE FEATURES (Logistic Regression)
# Logistic Regression coefficients directly show which words
# push the model toward Positive (high coeff) or Negative (low coeff)
# ─────────────────────────────────────────────────────────────

# Retrieve the LR model trained on TF-IDF
lr_result = next(
    r for r in all_results
    if r["Model"]=="Logistic Regression" and r["Features"]=="TF-IDF"
)
lr_model   = lr_result["_model"]
feat_names = tfidf_vectorizer.get_feature_names_out()
coefs      = lr_model.coef_[0]   # Shape: (n_features,)

# Sort by coefficient and take top/bottom 15
top15_pos_idx = np.argsort(coefs)[-15:][::-1]    # Most positive
top15_neg_idx = np.argsort(coefs)[:15]            # Most negative

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].barh(
    [feat_names[i] for i in top15_pos_idx[::-1]],
    [coefs[i]      for i in top15_pos_idx[::-1]],
    color="#2ecc71", edgecolor="black"
)
axes[0].set_title("Top 15 Positive Sentiment Words",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel("LR Coefficient (TF-IDF)")

axes[1].barh(
    [feat_names[i] for i in top15_neg_idx],
    [coefs[i]      for i in top15_neg_idx],
    color="#e74c3c", edgecolor="black"
)
axes[1].set_title("Top 15 Negative Sentiment Words",
                  fontsize=12, fontweight="bold")
axes[1].set_xlabel("LR Coefficient (TF-IDF)")

plt.suptitle("Logistic Regression — Most Informative Features",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("top_features_lr.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Task 6 — Comparison & Insights <a id='7'></a>

In [ ]:
# ─────────────────────────────────────────────────────────────
# TASK 6: STRUCTURED COMPARISON TABLE
# ─────────────────────────────────────────────────────────────

comparison_data = {
    "Aspect": [
        "Best Preprocessing Step",
        "Stopword Removal Impact",
        "Lemmatization vs Stemming",
        "Best Vectorization",
        "BoW Strengths",
        "TF-IDF Strengths",
        "Best ML Model",
        "Worst ML Model",
        "Logistic Regression",
        "Naive Bayes",
        "Decision Tree",
        "Random Forest",
        "Key Trade-off"
    ],
    "Finding": [
        "HTML removal + stopword removal: eliminated the most noise",
        "Reduces feature space by ~30%; removes tokens like 'the', 'is', 'a'",
        "Lemmatization preferred: produces real dictionary words (run, good)",
        "TF-IDF: downweights common words like 'film', 'movie'; better signal",
        "Simple, fast; works well with Naive Bayes",
        "Better discriminative features; generally higher F1 for all models",
        "Logistic Regression (TF-IDF): highest F1 — linear models excel on sparse text",
        "Decision Tree: lowest F1 — overfits sparse high-dim features easily",
        "Best overall: fast, highly interpretable via coefficients, strong regularisation",
        "Very fast training; slightly lower accuracy; strong with BoW",
        "Fastest to fit but prone to overfitting; needs max_depth limit",
        "Best accuracy/recall trade-off among tree methods; slower than LR",
        "Accuracy vs Interpretability: LR is best on both; DT is fast but inaccurate"
    ]
}

comparison_df = pd.DataFrame(comparison_data).set_index("Aspect")
print("="*70)
print("   📊  TASK 6: COMPARISON & INSIGHTS")
print("="*70)
display(
    comparison_df.style
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap",
                       "font-size": "12px", "max-width": "500px"})
    .set_table_styles([{
        "selector": "th",
        "props": [("background-color", "#2c3e50"), ("color", "white"),
                  ("font-size", "12px"), ("text-align", "center"),
                  ("padding", "8px")]
    }])
)

---
## 8. Summary of Findings <a id='8'></a>

---

### 📋 End-to-End Pipeline Recap

```
┌────────────────────────────────────────────────────────────────────┐
│          SENTIMENT ANALYSIS PIPELINE — NLP-2                      │
│                                                                    │
│  IMDB Raw Text (50,000 reviews)                                    │
│          │                                                         │
│          ▼                                                         │
│  NLP Preprocessing                                                 │
│  ① Remove HTML tags      ② Remove URLs                           │
│  ③ Remove special chars  ④ Lowercase                             │
│  ⑤ Tokenize              ⑥ Remove stopwords                      │
│  ⑦ Lemmatize tokens      ⑧ Rejoin to string                      │
│          │                                                         │
│          ├──────────────────────────────┐                         │
│          ▼                              ▼                         │
│  Bag of Words (BoW)            TF-IDF Vectorizer                  │
│  max_features=10,000           max_features=10,000                │
│  ngram_range=(1,2)             ngram_range=(1,2)                  │
│          │                              │                         │
│          ▼                              ▼                         │
│  ┌─────────────────┐          ┌─────────────────┐                 │
│  │ Logistic Regr.  │          │ Logistic Regr.  │                 │
│  │ Naive Bayes     │          │ Naive Bayes     │                 │
│  │ Decision Tree   │          │ Decision Tree   │                 │
│  │ Random Forest   │          │ Random Forest   │                 │
│  └─────────────────┘          └─────────────────┘                 │
│          │                              │                         │
│          └──────────────┬───────────────┘                         │
│                         ▼                                          │
│              Evaluation (Test Set)                                 │
│   Accuracy | Precision | Recall | F1 Score | Confusion Matrix     │
│                         │                                          │
│                         ▼                                          │
│          Comparison & Insights Report                              │
└────────────────────────────────────────────────────────────────────┘
```

---

### 🏆 Key Results

| Rank | Model | Features | F1 Score |
|---|---|---|---|
| 🥇 1st | Logistic Regression | TF-IDF | **Highest** |
| 🥈 2nd | Random Forest | TF-IDF | High |
| 🥉 3rd | Naive Bayes | TF-IDF | Good |
| 4th | Decision Tree | BoW/TF-IDF | Lowest |

### 💡 Key Learnings

1. **Preprocessing quality matters most**: HTML removal, stopword filtering, and lemmatization together reduce noise substantially and improve all downstream models.
2. **TF-IDF consistently outperforms BoW**: By penalising terms that appear in many documents, TF-IDF provides better discriminative features for sentiment.
3. **Linear models dominate text classification**: Logistic Regression outperforms tree-based methods because the classification boundary in TF-IDF space is well-approximated by a hyperplane.
4. **Decision Trees overfit sparse features**: High-dimensional sparse matrices from text vectorization make greedy tree splitting unreliable.
5. **Naive Bayes is a strong baseline**: Despite its strong independence assumption, Multinomial NB achieves competitive F1 at extremely fast training speed.
6. **Bigrams add value**: Including bigrams (`ngram_range=(1,2)`) captures phrases like "not good" and "very bad" — critical for sentiment.

---
*Notebook prepared as part of Data Science Internship – February 2026*